# Multimodal Counterfeit Detection

## Адаптация методов из статьи 2006.02231v1

Статья предлагает детектировать контрафактные приложения через комбинирование:
- **Content embeddings** (CNN fc7, 4096-dim)
- **Style embeddings** (матрица Грама из conv5_1, 131K-dim → random projection)
- **Text embeddings** (Doc2Vec PV-DM, 100-dim)

Финальная оценка схожести: `D = α·C_cos + β·S_cos + γ·T_cos`  
Оптимальные веса (grid search): β=5, γ=4.

### Адаптация к нашей задаче

| Статья | Наш датасет |
|---|---|
| Content embedding (CNN fc7) | Image embeddings (32-dim, предобученные) |
| Style embedding (Gram matrix) | **Не применимо** - нет сырых изображений/карт признаков |
| Text embedding (Doc2Vec PV-DM) | Doc2Vec на `name_rus + description + brand_name` |
| k-NN retrieval | Бинарная классификация (target: `resolution`) |
| Cosine distance fusion | **Late fusion** предсказанных вероятностей: `p = α·p_text + β·p_tab + γ·p_img` |

Табличные признаки (поведение продавца, продажи) - доменно-специфичный заменитель Style embeddings.

### Эксперименты
1. **Doc2Vec + LR** - метод статьи, применённый к тексту  
2. **Воспроизведение baseline'ов** - CatBoost (табл.), Image+LR  
3. **Late fusion с grid search** - адаптация `α·C + β·S + γ·T`  
4. **Feature fusion (CatBoost)** - конкатенация всех модальностей

## 0. Setup

In [ ]:
%%capture
!pip install catboost gensim

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import csv
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

SEED = 42
np.random.seed(SEED)

## 1. Загрузка данных и Seller-Based Split

Воспроизводим разбиение из baseline.ipynb **без изменений** для корректного сравнения метрик.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Путь к данным
DATA_PATH = '/content/drive//MyDrive/ozon_contest/ozon_train.csv'

df = pd.read_csv(DATA_PATH, encoding='utf-8', engine = 'python')
print(f'Загружено: {df.shape[0]:,} строк, {df.shape[1]} колонок')
print('Контрафакт', df["resolution"].mean():.2%)

In [ ]:
# Seller-based split — точная копия из baseline.ipynb
seller_class_counts = df.groupby('SellerID')['resolution'].nunique()
multi_class_sellers = seller_class_counts[seller_class_counts > 1].index

np.random.seed(SEED)
test_sellers = np.random.choice(
    multi_class_sellers,
    size=int(0.3 * len(multi_class_sellers)),
    replace=False
)
train_sellers = np.setdiff1d(df['SellerID'].unique(), test_sellers)

train_df = df[df['SellerID'].isin(train_sellers)].copy().reset_index(drop=True)
test_df  = df[df['SellerID'].isin(test_sellers)].copy().reset_index(drop=True)

print(f'Train: {train_df.shape[0]:,} строк, контрафакт: {train_df["resolution"].mean():.2%}')
print(f'Test:  {test_df.shape[0]:,} строк, контрафакт: {test_df["resolution"].mean():.2%}')

# Проверка воспроизводимости
assert train_df.shape[0] == 177380, f'Train size mismatch: {train_df.shape[0]}'
assert test_df.shape[0]  == 19818,  f'Test size mismatch: {test_df.shape[0]}'
print('✓ Разбиение воспроизведено корректно')

y_train = train_df['resolution'].values
y_test  = test_df['resolution'].values

## 2. Подготовка image embeddings

Критический шаг: `image_embeddings.csv` содержит эмбеддинги как для train, так и для test (ozon_test.csv).  
Необходимо отфильтровать **только `ozon_train` ItemID** до разбиения, иначе возникает утечка данных.

Для объектов без эмбеддинга используем нулевой вектор (zero imputation).

In [ ]:
def parse_embedding(s):
    return np.fromstring(s.strip().strip('[]'), sep=' ')

image_emb = pd.read_csv("image_embeddings.csv", on_bad_lines='skip')
image_emb.columns = [c.lower() for c in image_emb.columns]
image_emb['item_id'] = image_emb['image_name'].str.replace('.png', '', regex=False).astype(int)

print('Всего в image_emb', len(image_emb):,)

# Фильтрация: только train ItemID (КРИТИЧНО!)
image_emb_train = image_emb[image_emb['item_id'].isin(df['ItemID'])].copy()
print('После фильтрации (только ozon_train)', len(image_emb_train):,)
print('Отфильтровано (ozon_test items)', len(image_emb) - len(image_emb_train):,)


In [ ]:
EMB_DIM = 32

def build_image_matrix(items_df, emb_df, id_col='ItemID', emb_dim=EMB_DIM):
    merged = items_df[[id_col]].merge(
        emb_df[['item_id', 'embedding']],
        left_on=id_col, right_on='item_id', how='left'
    )
    missing_mask = merged['embedding'].isnull()
    n_missing = missing_mask.sum()
    if n_missing > 0:
        print(f'  Без эмбеддинга: {n_missing} → zero imputation')
        merged.loc[missing_mask, 'embedding'] = [' '.join(['0.0'] * emb_dim)] * n_missing
    return np.vstack(merged['embedding'].apply(parse_embedding))

print('Строим матрицы эмбеддингов...')
print('Train:')
X_img_train = build_image_matrix(train_df, image_emb_train)
print('Test:')
X_img_test  = build_image_matrix(test_df, image_emb_train)

print('X_img_train', X_img_train.shape)
print('X_img_test', X_img_test.shape)

img_scaler = StandardScaler()
X_img_train_sc = img_scaler.fit_transform(X_img_train)
X_img_test_sc  = img_scaler.transform(X_img_test)


## 3. Утилиты для оценки метрик

In [ ]:
results = {}  # Накапливаем все результаты для итоговой таблицы

def evaluate(y_true, y_prob, model_name=''):
    roc = roc_auc_score(y_true, y_prob)
    pr  = average_precision_score(y_true, y_prob)
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    mask = prec >= 0.9
    r90  = float(rec[mask].max()) if mask.any() else 0.0
    print(f'{model_name:<45}  ROC-AUC={roc:.4f}  PR-AUC={pr:.4f}  Recall@P≥0.9={r90:.4f}')
    results[model_name] = {'ROC-AUC': roc, 'PR-AUC': pr, 'Recall@P≥0.9': r90}
    return roc, pr, r90

## 4. Baseline A - Текст (TF-IDF + LR)

Воспроизведение из baseline.ipynb. Ожидаем ROC-AUC ≈ 0.9051, PR-AUC ≈ 0.5903.

In [ ]:
train_df['text'] = (train_df['name_rus'].fillna('') + ' ' +
                    train_df['description'].fillna('') + ' ' +
                    train_df['brand_name'].fillna(''))
test_df['text']  = (test_df['name_rus'].fillna('') + ' ' +
                    test_df['description'].fillna('') + ' ' +
                    test_df['brand_name'].fillna(''))

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=5)
X_tfidf_train = vectorizer.fit_transform(train_df['text'])
X_tfidf_test  = vectorizer.transform(test_df['text'])

lr_tfidf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_tfidf.fit(X_tfidf_train, y_train)
prob_tfidf_test = lr_tfidf.predict_proba(X_tfidf_test)[:, 1]

evaluate(y_test, prob_tfidf_test, 'TF-IDF + LR (baseline)')

## 5. Baseline B - Табличные признаки (CatBoost)

Воспроизведение из baseline.ipynb. Ожидаем ROC-AUC ≈ 0.9051, PR-AUC ≈ 0.6231.

In [ ]:
TARGET = 'resolution'
DROP_COLS = ['id', 'ItemID', 'SellerID', 'name_rus', 'description', 'brand_name', 'text', TARGET]

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]
cat_cols = train_df[feature_cols].select_dtypes(include=['object']).columns.tolist()
num_cols = train_df[feature_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Фичей всего: {len(feature_cols)}, категориальных: {len(cat_cols)}, числовых: {len(num_cols)}')

X_tab_train = train_df[feature_cols].copy()
X_tab_test  = test_df[feature_cols].copy()
X_tab_train[num_cols] = X_tab_train[num_cols].fillna(0)
X_tab_test[num_cols]  = X_tab_test[num_cols].fillna(0)

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

cb_model = CatBoostClassifier(
    iterations=700, depth=6, learning_rate=0.05,
    eval_metric='AUC', scale_pos_weight=scale_pos_weight,
    random_seed=SEED, verbose=100
)
cb_model.fit(X_tab_train, y_train, cat_features=cat_cols,
             eval_set=(X_tab_test, y_test), use_best_model=True)

prob_cb_test  = cb_model.predict_proba(X_tab_test)[:, 1]
prob_cb_train = cb_model.predict_proba(X_tab_train)[:, 1]  # нужны для fusion

evaluate(y_test, prob_cb_test, 'CatBoost tabular (baseline)')

## 6. Baseline C - Image Embeddings + LR

Воспроизведение из baseline.ipynb (с zero imputation для отсутствующих).  
Ожидаем ROC-AUC ≈ 0.8532, PR-AUC ≈ 0.4616.

In [ ]:
lr_img = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)
lr_img.fit(X_img_train_sc, y_train)
prob_img_test  = lr_img.predict_proba(X_img_test_sc)[:, 1]
prob_img_train = lr_img.predict_proba(X_img_train_sc)[:, 1]  # нужны для fusion

evaluate(y_test, prob_img_test, 'Image Embeddings + LR (baseline)')

## 7. Метод из статьи - Doc2Vec (PV-DM) Text Embeddings

Статья использует **Distributed Memory Model of Paragraph Vectors (PV-DM)** для текстовых эмбеддингов.

Параметры адаптированы для русского языка (больший `vector_size` для морфологии):
- `vector_size=200` (статья: 100; увеличено для лучшей передачи морфологии русского)
- `dm=1` → PV-DM (как в статье)
- `min_count=2` → убираем hapax legomena
- `epochs=20` → достаточно для корпуса ~177K документов

Важно: модель обучается только на `train_df`. Для `test_df` используется `infer_vector()` - это предотвращает утечку.

In [ ]:
print('Обучаем Doc2Vec (PV-DM)...')
print(f'Корпус: {len(train_df):,} документов')

tagged_train = [
    TaggedDocument(words=text.lower().split(), tags=[str(i)])
    for i, text in enumerate(train_df['text'])
]

d2v_model = Doc2Vec(
    vector_size=200,
    window=5,
    min_count=2,
    dm=1,           # PV-DM, как в статье
    epochs=20,
    seed=SEED,
    workers=4       # workers=1 для полной воспроизводимости, но медленнее
)

d2v_model.build_vocab(tagged_train)
print(f'Словарь: {len(d2v_model.wv):,} токенов')
d2v_model.train(tagged_train, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)
print('Doc2Vec обучен.')

In [ ]:
# Train: берём эмбеддинги документов по тегу
X_d2v_train = np.array([d2v_model.dv[str(i)] for i in range(len(train_df))])

# Test: infer_vector (steps=50 для стабильности коротких документов)
print('Инференс Doc2Vec на test...')
X_d2v_test = np.array([
    d2v_model.infer_vector(text.lower().split(), epochs=50)
    for text in test_df['text']
])

print('X_d2v_train', X_d2v_train.shape)
print('X_d2v_test', X_d2v_test.shape)

d2v_scaler = StandardScaler()
X_d2v_train_sc = d2v_scaler.fit_transform(X_d2v_train)
X_d2v_test_sc  = d2v_scaler.transform(X_d2v_test)

In [ ]:
lr_d2v = LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=SEED)
lr_d2v.fit(X_d2v_train_sc, y_train)
prob_d2v_test  = lr_d2v.predict_proba(X_d2v_test_sc)[:, 1]
prob_d2v_train = lr_d2v.predict_proba(X_d2v_train_sc)[:, 1]  # нужны для fusion

evaluate(y_test, prob_d2v_test, 'Doc2Vec PV-DM + LR (paper method)')

### Сравнение Doc2Vec vs TF-IDF

Doc2Vec ожидаемо слабее TF-IDF для данной задачи: TF-IDF захватывает точные совпадения торговых марок и артикулов ("xiaomi", "картридж hp"), которые являются сильными сигналами контрафакта. Однако Doc2Vec улавливает семантическое сходство и поведёт себя дополнительно при фьюжне.

## 8. Late Fusion с Grid Search весов

Адаптация из статьи:  
Статья: `D_combined = α·C_cos + β·S_cos + γ·T_cos` (grid search → β=5, γ=4)  
Мы: `p_combined = α·p_doc2vec + β·p_catboost + γ·p_image`

Ключевое требование: grid search весов должен проводиться на **валидационном сете**, а не на test.  
Иначе - утечка данных (weights overfit to test distribution).

In [ ]:
# Создаём валидационный сплит из train (80/20 stratified) для подбора весов
_, val_idx = train_test_split(
    np.arange(len(train_df)), test_size=0.2,
    stratify=y_train, random_state=SEED
)

y_val          = y_train[val_idx]
prob_d2v_val   = prob_d2v_train[val_idx]
prob_cb_val    = prob_cb_train[val_idx]
prob_img_val   = prob_img_train[val_idx]

print(f'Validation set: {len(val_idx):,} объектов, контрафакт: {y_val.mean():.2%}')

In [ ]:
def late_fusion_grid_search(p_text, p_tab, p_img, y_true, metric='pr_auc', step=0.1):
    """
    Grid search по весам (alpha, beta, gamma) с alpha + beta + gamma = 1.
    Оптимизируем PR-AUC (рекомендовано при дисбалансе классов).
    """
    best_score = -1
    best_weights = (1/3, 1/3, 1/3)

    grid = np.round(np.arange(0, 1 + step, step), 10)

    for alpha in grid:
        for beta in grid:
            gamma = round(1.0 - alpha - beta, 10)
            if gamma < -1e-9 or gamma > 1 + 1e-9:
                continue
            gamma = max(0.0, gamma)

            p_comb = alpha * p_text + beta * p_tab + gamma * p_img

            if metric == 'pr_auc':
                score = average_precision_score(y_true, p_comb)
            else:
                score = roc_auc_score(y_true, p_comb)

            if score > best_score:
                best_score = score
                best_weights = (alpha, beta, gamma)

    return best_weights, best_score


print('Grid search по весам (оптимизируем PR-AUC на validation)...')
best_weights, val_score = late_fusion_grid_search(
    prob_d2v_val, prob_cb_val, prob_img_val, y_val,
    metric='pr_auc', step=0.1
)
alpha, beta, gamma = best_weights
print(f'Лучшие веса: α(doc2vec)={alpha:.2f}, β(catboost)={beta:.2f}, γ(image)={gamma:.2f}')
print('Validation PR-AUC', val_score:.4f)

In [ ]:
# Применяем лучшие веса к test set
p_fusion_grid = alpha * prob_d2v_test + beta * prob_cb_test + gamma * prob_img_test
evaluate(y_test, p_fusion_grid, f'Late Fusion grid (α={alpha:.1f},β={beta:.1f},γ={gamma:.1f})')

# Для сравнения: веса из статьи (text:tabular:image = 1:5:4)
a0, b0, g0 = 1/10, 5/10, 4/10
p_fusion_paper = a0 * prob_d2v_test + b0 * prob_cb_test + g0 * prob_img_test
evaluate(y_test, p_fusion_paper, 'Late Fusion paper weights (1:5:4)')

### Анализ весов

Сравниваем найденные веса с пропорциями из статьи (1:5:4).  
В статье стиль имеет наибольший вес (β=5) - в нашем случае это аналог табличных признаков, которые несут поведенческие сигналы контрафакта.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

modalities = ['Doc2Vec\n(text)', 'CatBoost\n(tabular)', 'Image\nEmbeddings']
weights_grid  = [alpha, beta, gamma]
weights_paper = [1/10, 5/10, 4/10]

ax[0].bar(modalities, weights_grid, color=['#4C72B0', '#DD8452', '#55A868'])
ax[0].set_title('Веса (grid search, оптимиз. на val)')
ax[0].set_ylabel('Вес')
ax[0].set_ylim(0, 1)

ax[1].bar(modalities, weights_paper, color=['#4C72B0', '#DD8452', '#55A868'])
ax[1].set_title('Веса из статьи (адаптированные 1:5:4)')
ax[1].set_ylabel('Вес')
ax[1].set_ylim(0, 1)

plt.suptitle('Сравнение весов Late Fusion')
plt.tight_layout()
plt.show()

## 9. Feature Fusion - CatBoost на конкатенации всех признаков

Конкатенируем все три модальности в одну матрицу признаков и обучаем CatBoost.  
Это "early fusion" - модель сама учится взаимодействиям между модальностями.

In [ ]:
# Создаём DataFrame с эмбеддингами
d2v_col_names = [f'd2v_{i}' for i in range(X_d2v_train.shape[1])]
img_col_names = [f'img_{i}' for i in range(X_img_train.shape[1])]

def build_fused_df(X_tab, X_d2v, X_img, d2v_cols, img_cols):
    df_tab = X_tab.reset_index(drop=True)
    df_d2v = pd.DataFrame(X_d2v, columns=d2v_cols)
    df_img = pd.DataFrame(X_img, columns=img_cols)
    fused = pd.concat([df_tab, df_d2v, df_img], axis=1)
    # Проверка уникальности колонок
    assert len(set(fused.columns)) == len(fused.columns), 'Дублирующиеся колонки!'
    return fused

X_fused_train = build_fused_df(X_tab_train, X_d2v_train, X_img_train, d2v_col_names, img_col_names)
X_fused_test  = build_fused_df(X_tab_test,  X_d2v_test,  X_img_test,  d2v_col_names, img_col_names)

print('X_fused_train', X_fused_train.shape)
print('X_fused_test', X_fused_test.shape)
print('Категориальные колонки', cat_cols)

In [ ]:
cb_fused = CatBoostClassifier(
    iterations=700, depth=6, learning_rate=0.05,
    eval_metric='AUC', scale_pos_weight=scale_pos_weight,
    random_seed=SEED, verbose=100
)
cb_fused.fit(X_fused_train, y_train, cat_features=cat_cols,
             eval_set=(X_fused_test, y_test), use_best_model=True)

prob_fused_test = cb_fused.predict_proba(X_fused_test)[:, 1]
evaluate(y_test, prob_fused_test, 'Feature Fusion CatBoost (all modalities)')

### Feature importance для fused модели

Посмотрим, какие модальности CatBoost считает наиболее важными.

In [ ]:
fi = pd.DataFrame({
    'feature': X_fused_train.columns,
    'importance': cb_fused.get_feature_importance()
}).sort_values('importance', ascending=False)

# Группируем по модальности
def get_modality(name):
    if name.startswith('d2v_'):  return 'Doc2Vec'
    if name.startswith('img_'):  return 'Image'
    return 'Tabular'

fi['modality'] = fi['feature'].apply(get_modality)
modality_importance = fi.groupby('modality')['importance'].sum().sort_values(ascending=False)

print('Суммарная важность по модальностям:')
print(modality_importance.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modality_importance.plot(kind='bar', ax=axes[0], color=['#DD8452', '#4C72B0', '#55A868'])
axes[0].set_title('Суммарная важность по модальностям')
axes[0].set_ylabel('Суммарная важность')
axes[0].tick_params(axis='x', rotation=0)

# Топ-20 признаков в целом
top20 = fi.head(20)
colors = top20['modality'].map({'Tabular': '#DD8452', 'Doc2Vec': '#4C72B0', 'Image': '#55A868'})
axes[1].barh(range(20), top20['importance'].values, color=colors.values)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20['feature'].values, fontsize=9)
axes[1].set_title('Топ-20 признаков')
axes[1].invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#DD8452', label='Tabular'),
                   Patch(facecolor='#4C72B0', label='Doc2Vec'),
                   Patch(facecolor='#55A868', label='Image')]
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 10. Итоговое сравнение всех моделей

In [ ]:
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Model', 'ROC-AUC', 'PR-AUC', 'Recall@P≥0.9']
results_df = results_df.sort_values('PR-AUC', ascending=False).reset_index(drop=True)

# Отметим baseline и методы из статьи
def tag_model(name):
    if 'baseline' in name.lower(): return '📊 Baseline'
    if 'paper' in name.lower() or 'doc2vec' in name.lower() or 'pv-dm' in name.lower(): return '📄 Paper method'
    if 'fusion' in name.lower(): return '🔀 Fusion'
    return ''

results_df['Type'] = results_df['Model'].apply(tag_model)

print('=' * 90)
print(f'{"Model":<45}  {"ROC-AUC":>9}  {"PR-AUC":>9}  {"Recall@P≥0.9":>13}')
print('=' * 90)
for _, row in results_df.iterrows():
    print(f'{row["Model"]:<45}  {row["ROC-AUC"]:>9.4f}  {row["PR-AUC"]:>9.4f}  {row["Recall@P≥0.9"]:>13.4f}')
print('=' * 90)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['ROC-AUC', 'PR-AUC', 'Recall@P≥0.9']
colors_map = {'📊 Baseline': '#aec7e8', '📄 Paper method': '#ffbb78', '🔀 Fusion': '#98df8a', '': '#c5b0d5'}

for ax, metric in zip(axes, metrics):
    bars = ax.barh(
        results_df['Model'],
        results_df[metric],
        color=[colors_map.get(t, '#c5b0d5') for t in results_df['Type']]
    )
    ax.set_xlabel(metric)
    ax.set_title(metric)
    ax.invert_yaxis()
    # Линия лучшего baseline
    baseline_vals = results_df[results_df['Type'] == '📊 Baseline'][metric]
    if len(baseline_vals) > 0:
        ax.axvline(baseline_vals.max(), color='red', linestyle='--', alpha=0.5, label='Best baseline')
        ax.legend(fontsize=8)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in colors_map.items() if k]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Сравнение моделей: Baseline vs Paper methods vs Fusion', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Precision-Recall кривые для лучших моделей

In [ ]:
plt.figure(figsize=(10, 7))

models_to_plot = [
    ('TF-IDF + LR (baseline)',    prob_tfidf_test, '#aaaaaa', '--'),
    ('CatBoost tabular (baseline)', prob_cb_test, '#4C72B0', '--'),
    ('Image Embeddings + LR (baseline)', prob_img_test, '#cccccc', '--'),
    ('Doc2Vec PV-DM + LR (paper method)', prob_d2v_test, '#ff7f0e', '-'),
    (f'Late Fusion grid (α={alpha:.1f},β={beta:.1f},γ={gamma:.1f})', p_fusion_grid, '#2ca02c', '-'),
    ('Late Fusion paper weights (1:5:4)', p_fusion_paper, '#9467bd', '-.'),
    ('Feature Fusion CatBoost (all modalities)', prob_fused_test, '#d62728', '-'),
]

for label, probs, color, ls in models_to_plot:
    prec, rec, _ = precision_recall_curve(y_test, probs)
    pr_auc = average_precision_score(y_test, probs)
    plt.plot(rec, prec, label=f'{label} (PR-AUC={pr_auc:.4f})', color=color, linestyle=ls, lw=1.8)

plt.axvline(x=0.0, color='white')  # spacer
plt.axhline(y=0.9, color='grey', linestyle=':', alpha=0.7, label='Precision=0.9 (цель)')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall кривые')
plt.legend(loc='upper right', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Выводы

### Адаптация методов статьи 2006.02231v1

| Компонент статьи | Статус | Наша реализация |
|---|---|---|
| Content embeddings (VGG19 fc7) | ✅ Адаптировано | 32-dim предобученные image embeddings |
| Style embeddings (Gram matrix) | ❌ Неприменимо | Нет сырых feature maps; заменено табличными признаками |
| Text embeddings (Doc2Vec PV-DM) | ✅ Реализовано | Doc2Vec 200-dim на `name_rus + description + brand_name` |
| Weighted fusion с grid search | ✅ Адаптировано | `p = α·p_text + β·p_tab + γ·p_img` |
| k-NN retrieval | ❌ Неприменимо | Задача - бинарная классификация |

### Основные наблюдения

1. Doc2Vec vs TF-IDF: Doc2Vec (метод статьи) ожидаемо уступает TF-IDF в нашей задаче, так как точные совпадения торговых марок («xiaomi», «картридж hp») несут сильный дискриминативный сигнал.

2. Late Fusion с grid search: Подбор весов α/β/γ на validation-сете воспроизводит ключевую идею статьи. Оптимальные веса показывают, насколько каждая модальность дополняет остальные.

3. Feature Fusion (CatBoost): Конкатенация признаков позволяет модели самостоятельно обнаруживать взаимодействия между модальностями - потенциально сильнейший подход.

4. Recall@P≥0.9: Ключевая метрика для production-модерации. Мультимодальный фьюжн должен существенно улучшить её по сравнению с одномодальными baseline'ами (где она ≈ 0).